In [1]:
import glob
import os
import requests
import s3fs
import fiona
import glob
import netCDF4 as nc
import h5netcdf
import xarray as xr
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import hvplot.xarray
from datetime import datetime
# import earthaccess
# from earthaccess import Auth, DataCollections, DataGranules, Store

In [40]:
# Function to create map plot
def create_map_plot(ds_pixc):#data, lat, lon):
    fig, ax = plt.subplots(1,1, figsize=(8, 8))
    # plt.figure(figsize=(8, 6))
    ax.set_aspect('equal')
    plt.title(date)
    ax.set_ylabel('Latitude')
    ax.set_xlabel('Longitude')

    cmap = plt.cm.get_cmap('nipy_spectral')
    
    s=ax.scatter(x=ds_pixc.longitude, y=ds_pixc.latitude, c=ds_pixc.height, 
                   s=1, edgecolor='none', cmap=cmap,# cmap=plt.cm.nipy_spectral,
                vmin=-40, vmax=-29)

    bounds=list(range(-40, -29)) #[-40, -35, -30]
    
    cbar = fig.colorbar(mappable=s, ax=ax,   cmap=cmap, label='Water height (m)', boundaries=bounds) # norm=norm,
    cbar.set_label('Water height (m) to reference ellipsoid')

    ax = plt.gca()
    ax.set_xlim([xmin, xmax])
    ax.set_ylim([ymin, ymax])
    
    fig.tight_layout()

In [41]:
file_paths = glob.glob('../data/delaware/beta2/*.nc')#[0:3]
file_paths.sort()
xmin, xmax, ymin, ymax = -75.5, -75.4, 39.23, 39.33

# Loop through netCDF files
images = []
for file_path in file_paths:
    with xr.open_dataset(file_path, group = 'pixel_cloud', engine='h5netcdf') as ds_pixc:
    # with nc.Dataset(file_path, 'r') as dataset:

        ds_pixc.load()  # Load to memory

        date = datetime.strptime(file_path.split('_')[-4],'%Y%m%dT%H%M%S').strftime('%Y-%m-%d %H:%M:%S')
        
        # Subset variables
        ds_pixc = ds_pixc[['height','water_frac','cross_track','sig0','classification','classification_qual']]
        
        
        ds_pixc = ds_pixc.where((ds_pixc.latitude > ymin) & 
                                (ds_pixc.latitude <= ymax) & 
                                (ds_pixc.longitude > xmin) &
                                (ds_pixc.longitude <= xmax), drop=True)
        # Filter by cross-track to remove folded pixels?
        # ds_pixc = ds_pixc.where(ds_pixc.cross_track > 2000, drop=True)
        # ds_pixc = ds_pixc.sel(classification=np.isin(3,4))
        ds_pixc = ds_pixc.where((ds_pixc.classification==3) | (ds_pixc.classification==4), drop=True)
        
        # ds_pixc = ds_pixc.where((ds_pixc.classification in [3,4]).any() )
        ds_pixc = ds_pixc.where((ds_pixc.height < 20) & (ds_pixc.height > -50), drop=True)

        ds_pixc = ds_pixc.where(ds_pixc.sig0 > 10e-7, drop=True)
        # # Convert sigma0 from DN to decibels
        # ds_pixc['sig0_db'] = np.log10(ds_pixc.sig0)*10
        
        # ds_pixc = ds_pixc.where((ds_pixc.sig0_db > -20) & (ds_pixc.sig0_db < 40), drop=True)

        # Assuming you have latitude, longitude, and data variables in your netCDF file
        lat = ds_pixc.variables['latitude'][:]
        lon = ds_pixc.variables['longitude'][:]
        data = ds_pixc.variables['height'][:]

        # Create map plot
        create_map_plot(ds_pixc)#lat, lon, data)

        # Save plot to an image file
        image_file = file_path.replace('.nc', '.png')
        plt.savefig(image_file, dpi=400)
        plt.close()

        # Append image file to list of images
        images.append(image_file)
        
# Display content info
# ds_pixc.info

/var/folders/mc/t1rx478j0_v5wx1183167y8r0000gn/T/ipykernel_64989/987526095.py:10: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = plt.cm.get_cmap('nipy_spectral')
/var/folders/mc/t1rx478j0_v5wx1183167y8r0000gn/T/ipykernel_64989/987526095.py:10: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = plt.cm.get_cmap('nipy_spectral')
/var/folders/mc/t1rx478j0_v5wx1183167y8r0000gn/T/ipykernel_64989/987526095.py:10: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = plt.cm.get

In [42]:
import imageio
# Create GIF animation from images
gif_file = "../output/figures/delaware_swot_beta_v3.gif"
with imageio.get_writer(gif_file, duration=12, loop=0, fps=2) as writer:  #  mode='I',  fps
    for image in images:
        writer.append_data(imageio.imread(image))

/var/folders/mc/t1rx478j0_v5wx1183167y8r0000gn/T/ipykernel_64989/2970506288.py:6: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  writer.append_data(imageio.imread(image))


In [ ]:
with xr.open_dataset(file_path, engine='h5netcdf') as ds_pixc:  # group = 'pixel_cloud', 

    ds_pixc.load()  # Load to memory
    # ds_pixc.info()
    ds_pixc.attrs
ds_pixc.attrs['ellipsoid_semi_major_axis']
ds_pixc.attrs['ellipsoid_flattening']
